In [47]:
import pandas as pd
import geopandas as gdp
import ee
import geemap
import os
from pathlib import Path


In [53]:

print("cwd:", Path.cwd())

matches = list(Path(".").rglob("P04-00-DHS_Boundaries_v1.0.csv"))
print("matches:", matches)

csv_path = matches[0]
print("using:", csv_path)
print("absolute:", csv_path.resolve())
print("exists:", csv_path.exists())

dhs_filter = pd.read_csv(csv_path.resolve(), encoding="utf-8-sig")
print(dhs_filter.head())
print(dhs_filter.columns.tolist())

cwd: /Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs
matches: [PosixPath('data/P04-00-DHS_Boundaries_v1.0.csv')]
using: data/P04-00-DHS_Boundaries_v1.0.csv
absolute: /Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/data/P04-00-DHS_Boundaries_v1.0.csv
exists: True
  Country Name  Download Boundaries Admin level  Standard DHS Other     Year  Regions  Phase  \
0  Afghanistan                  1.0           1           1.0   NaN     2015     34.0  DHS-7   
1      Albania                  1.0           1           1.0   NaN  2017-18     12.0  DHS-7   
2       Angola                  1.0           1           1.0   NaN  2023-24     18.0  DHS-8   
3       Angola                  0.0           1           1.0   NaN  2015-16     18.0  DHS-7   
4      Armenia                  1.0           1           1.0   NaN  2015-16     11.0  DHS-7   

   GPS*  Back-up  Downloaded  Unnamed: 11  
0   0.0      0.0         NaN          NaN  
1   1.0      0.0        

In [11]:

ee.Authenticate(auth_mode='localhost')

True

In [43]:
ee.Initialize(project="ee-estheregreenwood")

In [111]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# DHS_boundary data are in directory with per country subdirectories containing the DHS region shapefiles (in subdir 'shps')
# MICS_boundary data are in directory with per country subdirectories containing the MICS region shapefiles (in subdir 'Shapefiles'); one shapefile per admin level
# WFP_boundary data includes one shapefile for Pakistan with country subdirectories at level 2 (in subdir 'Shapefiles'); 
# GADM_boundary data includes one shapefile with 130 low and middle income country (World Bank income groups 2024) subdirectories at level 1

DHS_boundaries = 'data/DHS_boundaries'
MICS_boundaries = 'data/MICS_boundaries'
WFP_boundaries = 'data/PakistanWFP2024Boundaries_2'
GADM_lmic_shp = 'data/GADM36_boundaries/gadm_lmics_final.shp'

# Build allowed (country, year) set from download filter CSV
# Directories use either start or end year of "YYYY-YY" ranges, so we expand both
import re

def years_from_csv_year(year_str):
    year_str = str(year_str).strip()
    years = set(int(y) for y in re.findall(r'\d{4}', year_str))
    m = re.match(r'(\d{4})-(\d{2})$', year_str)
    if m:
        start = int(m.group(1))
        years.add((start // 100) * 100 + int(m.group(2)))
    return years

dhs_filter = pd.read_csv('data/P04-00-DHS_Boundaries_v1.0.csv', encoding='utf-8-sig')
dhs_filter = dhs_filter[dhs_filter['Download Boundaries'] == 1]
included_dhs_studies = set()
for _, row in dhs_filter.iterrows():
    for yr in years_from_csv_year(row['Year']):
        included_dhs_studies.add((row['Country Name'], yr))

# Load all DHS shapefiles
dhs_gdfs = []
for country_dir in Path(DHS_boundaries).iterdir():
    shp = country_dir / 'shps' / 'sdr_subnational_boundaries.shp'
    if shp.exists():
        # Directory format: "Country Name YYYY TYPE"
        parts = country_dir.name.rsplit(' ', 2)
        dir_country, dir_year = ' '.join(parts[:-2]), int(parts[-2])
        if (dir_country, dir_year) in included_dhs_studies:
            dhs_gdfs.append(gdp.read_file(shp))
dhs_gdf = gdp.GeoDataFrame(pd.concat(dhs_gdfs, ignore_index=True))

# Load all MICS shapefiles — track admin_level from filename suffix (1/2/3)
mics_gdfs = []
for country_dir in Path(MICS_boundaries).iterdir():
    shapefiles_dir = country_dir / 'Shapefiles'
    if shapefiles_dir.exists():
        for shp in sorted(shapefiles_dir.glob('mics_boundaries*.shp')):
            suffix = shp.stem.replace('mics_boundaries', '')
            level = int(suffix) if suffix.isdigit() else 1
            gdf = gdp.read_file(shp)
            gdf['admin_level'] = level
            mics_gdfs.append(gdf)
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore', FutureWarning)
    mics_gdf = gdp.GeoDataFrame(pd.concat([gdf for gdf in mics_gdfs if not gdf.empty], ignore_index=True))

# Load WFP file shapefile for Pakistan

wfp_gdfs = []
shapefiles_dir = Path(WFP_boundaries) / 'Shapefiles'
if shapefiles_dir.exists():
    for shp in sorted(shapefiles_dir.glob('WFP_boundaries*.shp')):
        suffix = shp.stem.replace('WFP_boundaries', '')
        level = int(suffix) if suffix.isdigit() else 1
        gdf = gdp.read_file(shp)
        gdf['admin_level'] = level
        wfp_gdfs.append(gdf)

with warnings.catch_warnings():
    warnings.simplefilter('ignore', FutureWarning)
    wfp_gdf = gdp.GeoDataFrame(pd.concat([gdf for gdf in wfp_gdfs if not gdf.empty], ignore_index=True))

# Load merged GADM shapefile

gadm_lmic_gdf = gpd.read_file(GADM_lmic_shp)

# Drop null/empty geometries and force 2D
dhs_gdf = dhs_gdf[~(dhs_gdf.geometry.isna() | dhs_gdf.geometry.is_empty)].copy()
dhs_gdf['geometry'] = dhs_gdf['geometry'].force_2d()
mics_gdf = mics_gdf[~(mics_gdf.geometry.isna() | mics_gdf.geometry.is_empty)].copy()
mics_gdf['geometry'] = mics_gdf['geometry'].force_2d()
wfp_gdf = wfp_gdf[~(wfp_gdf.geometry.isna() | wfp_gdf.geometry.is_empty)].copy()
wfp_gdf['geometry'] = wfp_gdf['geometry'].force_2d()
gadm_lmic_gdf = gadm_lmic_gdf[~(gadm_lmic_gdf.geometry.isna() | gadm_lmic_gdf.geometry.is_empty)].copy()
gadm_lmic_gdf['geometry'] = gadm_lmic_gdf.geometry.force_2d()

# Add analysis year column to both dataframes (DHS uses 'SVYYEAR', MICS uses 'SVYYEARS')
dhs_gdf['analysis_year'] = dhs_gdf['SVYYEAR'].astype(int)
dhs_gdf['admin_level'] = 1  # DHS always one level per survey
mics_gdf['analysis_year'] = mics_gdf['SVYYEARS']
wfp_gdf['analysis_year'] = 2022
gadm_lmic_gdf['analysis_year'] = 2024

gadm_lmic_gdf['admin_level'] = 1
gadm_lmic_gdf['source'] = 'GADM'

print('DHS size:', len(dhs_gdf))
print('MICS size:', len(mics_gdf))
print('WFP size:', len(wfp_gdf))
print('GADM size:', len(gadm_lmic_gdf))

/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/pyogrio/raw.py:198: RuntimeWarning: data/MICS_boundaries/GambiaMICS2018Boundaries_2/Shapefiles/mics_boundaries2.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(
/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(


DHS size: 690
MICS size: 380
WFP size: 160
GADM size: 2310


In [ ]:
# Defining adminstrative areas reported in household surveys
# Define island groups for Kiribati
noGil = ee.Geometry.Polygon(
        [[[173.76214858993802, 1.688437284048748],
          [173.89398452743802, 3.2689380495714633],
          [172.53167983993802, 4.277476168442425],
          [172.05246629783107, 3.029716351603881],
          [172.84548591405644, 1.4656846206052414],
          [173.12175664737128, 1.3820186787491144],
          [173.1354466421833, 1.3865663776048271],
          [173.14021024539375, 1.3954043333854713]]], None, False)
souTara = ee.Geometry.Polygon(
    [[[173.14024407183683, 1.3954382342837202],
        [173.13552338397062, 1.3865573758520389],
        [173.12591034686125, 1.382867734037598],
        [172.9929535267801, 1.3664672516799288],
        [172.8940765736551, 1.3946115454160364],
        [172.8940765736551, 1.317042332174653],
        [173.04582523088166, 1.304685947700206],
        [173.21679996232697, 1.3609756435068756],
        [173.17560123185822, 1.406967470897869]]], None, False)
cenGil = ee.Geometry.Polygon(
    [[[172.75257236267254, 1.106559466163889],
        [172.75257236267254, -0.947771870448704],
        [174.70813876892254, -0.947771870448704],
        [174.70813876892254, 1.106559466163889]]], None, False)
souGil = ee.Geometry.Polygon(
    [[[174.44446689392254, -1.0356493216635867],
        [174.44446689392254, -2.858249945133194],
        [177.15808994079754, -2.858249945133194],
        [177.15808994079754, -1.0356493216635867]]], None, False)
lineAndPhoenix =  ee.Geometry.Polygon(
    [[[178.7565749612447, 8.140225397293124],
        [178.7565749612447, -20.88502241263751],
        [223.75657496124472, -20.88502241263751],
        [223.75657496124472, 8.140225397293124]]], None, False)

# Define the countries to sample from level 0, 2, 3 or 1
countryLevel0 = ['Kiribati','Tonga']
countryLevel1 = ['Gambia','Iraq','Togo', 'Chad','Georgia','Ghana','Guinea-Bissau','Guyana','Laos','Lesotho','Nigeria','Paraguay','Sierra Leone','Suriname','Tonga','Zimbabwe','Kongo','Benin','Honduras', 'Tuvalu','Vanuatu' ]
countryLevel2 = ['Bangladesh','Gambia','Iraq','Kosovo','Madagascar','Palestina','Togo', 'São Tomé and Príncipe']
countryLevel3 = ['Pakistan']

# Load the GADM datasets 
GADM_level0 = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level0') \
                    .filter(ee.Filter.inList('NAME_0',['Kiribati']))

GADM_level1_prep = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level1')

GADM_level1 = GADM_level1_prep.filter(ee.Filter.inList('NAME_0',countryLevel1)) \
                                  .map(lambda f: f.set('HH7',f.getString('NAME_1')))

GADM_level2_prep = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level2')
GADM_level2 = GADM_level2_prep.filter(ee.Filter.inList('NAME_0',countryLevel2)) \
                                  .map(lambda f: f.set('HH7',f.getString('NAME_2')))
                    
GADM_level3_prep = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level3')
GADM_level3 = GADM_level3_prep \
                    .filter(ee.Filter.inList('NAME_0',countryLevel3)) \
                    .map(lambda f: f.set('HH7',f.getString('NAME_3')))



# Tunisia
TUNISIA_District_Tunis = ["Ariana","Ben Arous (Tunis Sud)","Bizerte","Manubah","Nabeul","Tunis","Zaghouan"]
TUNISIA_NORD_OUEST = ["Béja","Jendouba","Le Kef","Siliana"]
TUNISIA_CENTRE_EST = ["Mahdia","Monastir","Sousse"]
TUNISIA_CENTRE_OUEST = ["Kairouan","Kassérine","Sidi Bou Zid"]
TUNISIA_SUD_EST = ["Gabès","Médenine","Sfax","Tataouine"]
TUNISIA_SUD_OUEST = ["Gafsa","Kebili","Tozeur"]



# Define Kiribati 
NORTHERN_GILBERT = ee.Feature(GADM_level0.geometry().intersection(noGil, 10)).set('NAME_1','NORTHERN GILBERT').set('NAME_0','Kiribati').set('HH7','NORTHERN_GILBERT')
SOUTH_TARAWA = ee.Feature(GADM_level0.geometry().intersection(souTara, 10)).set('NAME_1','SOUTH TARAWA').set('NAME_0','Kiribati').set('HH7','SOUTH_TARAWA')
CENTRAL_GILBERT = ee.Feature(GADM_level0.geometry().intersection(cenGil, 10)).set('NAME_1','CENTRAL GILBERT').set('NAME_0','Kiribati').set('HH7','CENTRAL_GILBERT')
SOUTHERN_GILBERT = ee.Feature(GADM_level0.geometry().intersection(souGil, 10)).set('NAME_1','SOUTHERN GILBERT').set('NAME_0','Kiribati').set('HH7','SOUTHERN_GILBERT')
LINE_AND_PHOENIX = ee.Feature(GADM_level0.geometry().intersection(lineAndPhoenix, 10)).set('NAME_1','LINE AND PHOENIX GROUP').set('NAME_0','Kiribati').set('HH7','LINE_AND_PHOENIX')
GADM_level0 = ee.FeatureCollection([NORTHERN_GILBERT, SOUTH_TARAWA, CENTRAL_GILBERT, SOUTHERN_GILBERT, LINE_AND_PHOENIX])

##/ Define the new regions for Level 1

# Vietnam (Nour)
NORTHERN_MIDLAND = ['Hà Giang','Cao Bằng','Lào Cai','Lai Châu','Bắc Kạn','Tuyên Quang','Lạng Sơn','Yên Bái','Điện Biên','Thái Nguyên','Bắc Giang','Phú Thọ','Sơn La','Hòa Bình']
RED_RIVER_DELTA = ['Vĩnh Phúc','Quảng Ninh','Bắc Ninh','Hà Nội','Hải Dương','Hưng Yên','Hải Phòng','Hà Nam','Thái Bình','Nam Định','Ninh Bình']
NORTH_CENTRAL_AND_COASTAL = ['Thanh Hóa','Nghệ An','Hà Tĩnh','Quảng Bình','Quảng Trị','Thừa Thiên - Huế','Đà Nẵng','Quảng Nam','Quảng Ngãi','Bình Định','Phú Yên','Khánh Hòa','Ninh Thuận','Bình Thuận']
CENTRAL_HIGHLANDS = ['Kon Tum','Gia Lai','Đắk Lắk','Đăk Nông','Lâm Đồng']
SOUTH_EAST = ['Bình Phước','Tây Ninh','Bình Dương','Đồng Nai','Hồ Chí Minh city','Bà Rịa - Vũng Tàu']
MEKONG_DELTA = ['Long An','Đồng Tháp','An Giang','Tiền Giang','Bến Tre','Cần Thơ','Vĩnh Long','Kiên Giang','Trà Vinh','Hậu Giang','Sóc Trăng','Bạc Liêu','Cà Mau']

# Dominican Republic (Nour)
OZAMA = ['Distrito Nacional','Santo Domingo']
VALDESIA = ['Azua','Peravia','San José de Ocoa','San Cristóbal']
ENRIQUILLO = ['Barahona','Bahoruco','Independencia','Pedernales']
CIBAO_NORDESTE = ['Duarte','Salcedo','María Trinidad Sánchez','Samaná']
CIBAO_NOROESTE = ['Dajabón','Monte Cristi','Santiago Rodríguez','Valverde']
CIBAO_NORTE = ['Espaillat','Puerto Plata','Santiago']
CIBAO_SUR = ['La Vega','Monseñor Nouel','Sánchez Ramírez']
EL_VALLE = ['La Estrelleta','San Juan']
HIGUAMO = ['Hato Mayor','Monte Plata','San Pedro de Macorís']
YUMA = ['El Seybo','La Romana','La Altagracia']

# Algeria 
NORD_CENTRE = ['Alger','Blida','Boumerdès','Tipaza','Bouira','Médéa','Tizi Ouzou','Béjaïa','Chlef','Aïn Defla']
NORD_EST = ['Annaba','Constantine','Skikda','Jijel','Mila','Souk Ahras','El Tarf','Guelma']
NORD_OUEST = ['Oran','Tlemcen','Mostaganem','Aïn Témouchent','Relizane','Sidi Bel Abbès','Mascara']
HAUT_PLATEAU_CENTRE = ['Djelfa','Laghouat',"M'Sila"]
HAUT_PLATEAU_EST = ['Sétif','Batna','Khenchela','Bordj Bou Arréridj','Oum el Bouaghi','Tébessa']
HAUT_PLATEAU_OUEST = ['Saïda','Tissemsilt','Naâma','El Bayadh','Tiaret']
SUD = ['Béchar','Tindouf','Adrar','Ghardaïa','Biskra','El Oued','Ouargla','Tamanghasset','Illizi']

# Central African Republic 
REGION_1 = ["Ombella-M'Poko","Lobaye"]
REGION_2 = ["Mambéré-Kadéï","Nana-Mambéré","Sangha-Mbaéré"]
REGION_3 = ["Ouham-Pendé","Ouham"]
REGION_4 = ["Kémo","Nana-Grébizi","Ouaka"]
REGION_5 = ["Bamingui-Bangoran","Haute-Kotto","Vakaga"]
REGION_6 = ["Basse-Kotto","Mbomou","Haut-Mbomou"]
REGION_7 = ["Bangui"]

# Mongolia
WESTERN = ["Bayan-Ölgiy","Govi-Altay","Dzavhan","Hovd","Uvs"]
EASTERN = ["Dornod","Sühbaatar","Hentiy"]
ULAANBAATAR = ["Ulaanbaatar"]
CENTRAL = ["Töv","Ömnögovi","Selenge","Orhon","Darhan-Uul","Govisümber","Dundgovi","Dornogovi"]
KHANGAI = ["Bayanhongor","Hövsgöl","Arhangay","Övörhangay","Bulgan"]


# Concatenate the districts into regions
NORD_CENTRE = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",NORD_CENTRE))
NORD_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",NORD_EST))
NORD_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",NORD_OUEST))
HAUT_PLATEAU_CENTRE = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",HAUT_PLATEAU_CENTRE))
HAUT_PLATEAU_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",HAUT_PLATEAU_EST))
HAUT_PLATEAU_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",HAUT_PLATEAU_OUEST))
SUD = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",SUD))

REGION_1 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_1))
REGION_2 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_2))
REGION_3 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_3))
REGION_4 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_4))
REGION_5 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_5))
REGION_6 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_6))
REGION_7 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_7))

WESTERN = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",WESTERN))
EASTERN = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",EASTERN))
ULAANBAATAR = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",ULAANBAATAR))
CENTRAL = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",CENTRAL))
KHANGAI = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",KHANGAI))
           
TUNISIA_District_Tunis = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_District_Tunis))
TUNISIA_NORD_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_NORD_OUEST))
TUNISIA_CENTRE_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_CENTRE_EST))
TUNISIA_CENTRE_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_CENTRE_OUEST))
TUNISIA_SUD_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_SUD_EST))
TUNISIA_SUD_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_SUD_OUEST))
                          
# Final FCs
Algeria = ee.FeatureCollection([
    ee.Feature(NORD_CENTRE.union().geometry()).set('NAME_0','Algeria').set('NAME_1','NORD CENTRE').set('NAME_1s',NORD_CENTRE.aggregate_array('NAME_1').join('|')).set('GID_1s',NORD_CENTRE.aggregate_array('GID_1').join('|')).set('HH7','NORD_CENTRE'),
    ee.Feature(NORD_EST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','NORD EST').set('NAME_1s',NORD_EST.aggregate_array('NAME_1').join('|')).set('GID_1s',NORD_EST.aggregate_array('GID_1').join('|')).set('HH7','NORD_EST'),
    ee.Feature(NORD_OUEST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','NORD OUEST').set('NAME_1s',NORD_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',NORD_OUEST.aggregate_array('GID_1').join('|')).set('HH7','NORD_OUEST'),
    ee.Feature(HAUT_PLATEAU_CENTRE.union().geometry()).set('NAME_0','Algeria').set('NAME_1','HAUT PLATEAU CENTRE').set('NAME_1s',HAUT_PLATEAU_CENTRE.aggregate_array('NAME_1').join('|')).set('GID_1s',HAUT_PLATEAU_CENTRE.aggregate_array('GID_1').join('|')).set('HH7','HAUT_PLATEAU_CENTRE'),
    ee.Feature(HAUT_PLATEAU_EST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','HAUT PLATEAU EST').set('NAME_1s',HAUT_PLATEAU_EST.aggregate_array('NAME_1').join('|')).set('GID_1s',HAUT_PLATEAU_EST.aggregate_array('GID_1').join('|')).set('HH7','HAUT_PLATEAU_EST'),
    ee.Feature(HAUT_PLATEAU_OUEST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','HAUT PLATEAU OUEST').set('NAME_1s',HAUT_PLATEAU_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',HAUT_PLATEAU_OUEST.aggregate_array('GID_1').join('|')).set('HH7','HAUT_PLATEAU_OUEST'),
    ee.Feature(SUD.union().geometry()).set('NAME_0','Algeria').set('NAME_1','SUD').set('NAME_1s',SUD.aggregate_array('NAME_1').join('|')).set('GID_1s',SUD.aggregate_array('GID_1').join('|')).set('HH7','SUD'),
    ])
CAR = ee.FeatureCollection([
    ee.Feature(REGION_1.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_1').set('NAME_1s',REGION_1.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_1.aggregate_array('GID_1').join('|')).set("HH7","REGION_1"),
    ee.Feature(REGION_2.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_2').set('NAME_1s',REGION_2.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_2.aggregate_array('GID_1').join('|')).set("HH7","REGION_2"),
    ee.Feature(REGION_3.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_3').set('NAME_1s',REGION_3.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_3.aggregate_array('GID_1').join('|')).set("HH7","REGION_3"),
    ee.Feature(REGION_4.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_4').set('NAME_1s',REGION_4.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_4.aggregate_array('GID_1').join('|')).set("HH7","REGION_4"),
    ee.Feature(REGION_5.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_5').set('NAME_1s',REGION_5.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_5.aggregate_array('GID_1').join('|')).set("HH7","REGION_5"),
    ee.Feature(REGION_6.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_6').set('NAME_1s',REGION_6.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_6.aggregate_array('GID_1').join('|')).set("HH7","REGION_6"),
    ee.Feature(REGION_7.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_7').set('NAME_1s',REGION_7.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_7.aggregate_array('GID_1').join('|')).set("HH7","REGION_7")
    ])
Mongolia = ee.FeatureCollection([
    ee.Feature(WESTERN.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','WESTERN').set('NAME_1s',WESTERN.aggregate_array('NAME_1').join('|')).set('GID_1s',WESTERN.aggregate_array('GID_1').join('|')).set('HH7','WESTERN'),
    ee.Feature(EASTERN.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','EASTERN').set('NAME_1s',EASTERN.aggregate_array('NAME_1').join('|')).set('GID_1s',EASTERN.aggregate_array('GID_1').join('|')).set('HH7','EASTERN'),
    ee.Feature(ULAANBAATAR.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','ULAANBAATAR').set('NAME_1s',ULAANBAATAR.aggregate_array('NAME_1').join('|')).set('GID_1s',ULAANBAATAR.aggregate_array('GID_1').join('|')).set('HH7','ULAANBAATAR'),
    ee.Feature(CENTRAL.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','CENTRAL').set('NAME_1s',CENTRAL.aggregate_array('NAME_1').join('|')).set('GID_1s',CENTRAL.aggregate_array('GID_1').join('|')).set('HH7','CENTRAL'),
    ee.Feature(KHANGAI.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','KHANGAI').set('NAME_1s',KHANGAI.aggregate_array('NAME_1').join('|')).set('GID_1s',KHANGAI.aggregate_array('GID_1').join('|')).set('HH7','KHANGAI')
    ])
Tunisia = ee.FeatureCollection([
    ee.Feature(TUNISIA_District_Tunis.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_District_Tunis').set('NAME_1s',TUNISIA_District_Tunis.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_District_Tunis.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_District_Tunis'),
    ee.Feature(TUNISIA_NORD_OUEST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_NORD_OUEST').set('NAME_1s',TUNISIA_NORD_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_NORD_OUEST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_NORD_OUEST'),
    ee.Feature(TUNISIA_CENTRE_EST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_CENTRE_EST').set('NAME_1s',TUNISIA_CENTRE_EST.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_CENTRE_EST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_CENTRE_EST'),
    ee.Feature(TUNISIA_CENTRE_OUEST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_CENTRE_OUEST').set('NAME_1s',TUNISIA_CENTRE_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_CENTRE_OUEST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_CENTRE_OUEST'),
    ee.Feature(TUNISIA_SUD_EST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_SUD_EST').set('NAME_1s',TUNISIA_SUD_EST.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_SUD_EST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_SUD_EST'),
    ee.Feature(TUNISIA_SUD_OUEST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_SUD_OUEST').set('NAME_1s',TUNISIA_SUD_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_SUD_OUEST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_SUD_OUEST')
    ])    
GADM_level1 = GADM_level1.merge(Algeria).merge(CAR).merge(Tunisia).merge(Mongolia)

# Add the missing properties 
GADM_level3 = GADM_level3.map(lambda f: f.set('NAME_1s', 'NaN').set('GID_1s', 'NaN'))
GADM_level0 = GADM_level0.map(lambda f: f.set(ee.Dictionary.fromLists(                                                                
      p := GADM_level3.first().propertyNames().removeAll(GADM_level0.first().propertyNames()),                                          
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))             
GADM_level2 = GADM_level2.map(lambda f: f.set(ee.Dictionary.fromLists(                                                                
      p := GADM_level3.first().propertyNames().removeAll(GADM_level2.first().propertyNames()),
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))                                                                                                                                   
GADM_level1 = GADM_level1.map(lambda f: f.set(ee.Dictionary.fromLists(                                                                
      p := GADM_level3.first().propertyNames().removeAll(GADM_level1.first().propertyNames()),
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))

# Merge the data 
GADM_toSample = GADM_level2.merge(GADM_level0).merge(GADM_level3).merge(GADM_level1)
# print(GADM_toSample)

##/ Define the new regions for Level 3
Pakistan_Gujranwala = ee.FeatureCollection(                        
      ee.Feature(GADM_level3_prep.filterMetadata('NAME_3', 'contains', 'Gujranwala').geometry())
      .set('NAME_0', 'Pakistan')                                                                                                        
      .set('NAME_3', 'Gujranwala')                                                                                                      
      .set('HH7', 'Gujranwala')                                                                                                         
  ).map(lambda f: f.set(ee.Dictionary.fromLists(                                                                                        
      p := GADM_level3_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_3']),                                                    
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))             
                                                                                                                                        
REGIAO_NORTE_OESTE = ee.FeatureCollection(
      ee.Feature(GADM_level2_prep.filterMetadata('NAME_0', 'contains', 'São Tomé and Príncipe')
      .filter(ee.Filter.inList('NAME_2', ['Lembá', 'Lobata'])).geometry())                                                              
      .set('NAME_0', 'São Tomé and Príncipe')
      .set('NAME_2', 'REGIÃO NORTE OESTE')                                                                                              
      .set('HH7', 'REGIAO_NORTE_OESTE')
  ).map(lambda f: f.set(ee.Dictionary.fromLists(                                                                                        
      p := GADM_level2_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_2']),                                                    
      ee.List.repeat('NaN', p.size())
  )))                                                                                                                                   
                  
REGIAO_SUL_ESTE = ee.FeatureCollection(                                                                                               
      ee.Feature(GADM_level2_prep.filterMetadata('NAME_0', 'contains', 'São Tomé and Príncipe')
      .filter(ee.Filter.inList('NAME_2', ['Cantagalo', 'Caué'])).geometry())                                                            
      .set('NAME_0', 'São Tomé and Príncipe')                                                                                           
      .set('NAME_2', 'REGIÃO SUL ESTE')
      .set('HH7', 'REGIAO_SUL_ESTE')                                                                                                    
  ).map(lambda f: f.set(ee.Dictionary.fromLists(                                                                                        
      p := GADM_level2_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_2']),
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))             
                                                                                                                                        
Principe = ee.FeatureCollection(
      ee.Feature(GADM_level1_prep.filterMetadata('NAME_0', 'contains', 'São Tomé and Príncipe')
      .filter(ee.Filter.inList('NAME_1', ['Príncipe'])).geometry())                                                                     
      .set('NAME_0', 'São Tomé and Príncipe')
      .set('NAME_1', 'Príncipe')                                                                                                        
      .set('HH7', 'Príncipe')                                                                                                           
  ).map(lambda f: f.set(ee.Dictionary.fromLists(
      p := GADM_level1_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_1']),                                                    
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))

# Add them to the to sampling data 
GADM_toSample = GADM_toSample.merge(Pakistan_Gujranwala).merge(REGIAO_NORTE_OESTE).merge(REGIAO_SUL_ESTE).merge(Principe)

# filter countries from GADM that are in MICS or DHS datasets
countries_in_DHS = dhs_gdf['CNTRYNAMEE'].unique().tolist()
countries_in_MICS = mics_gdf['CNTRYNAME'].unique().tolist()
countries_to_exclude = set(countries_in_DHS + countries_in_MICS)
GADM_toSample = GADM_toSample.filter(ee.Filter.inList('NAME_0', list(countries_to_exclude)).Not())

In [113]:
# Below list of coutntries that need manual assignment of survey year 
analysisYearByCountry = {
    'Algeria': 2018,
    'Bangladesh': 2019,
    'Central African Republic': 2018,
    'Chad': 2019,
    'Gambia': 2018,
    'Georgia': 2018,
    'Ghana': 2017,
    'Guinea-Bissau': 2018,
    'Guyana': 2019,
    'Iraq': 2018,
    'Kiribati': 2018,
    'Kosovo': 2019,
    'Laos': 2023,
    'Lesotho': 2018,
    'Madagascar': 2018,
    'Mongolia': 2018,
    'Nigeria': 2016,
    'Pakistan': 2017,
    'Palestina': 2019,
    'Paraguay': 2016,
    'Sierra Leone': 2017,
    'Suriname': 2018,
    'São Tomé and Príncipe': 2019,
    'Togo': 2017,
    'Tonga': 2019,
    'Tunisia': 2023,
    'Zimbabwe': 2018,
}

# Get year from dictionary and assign to each feature
year_dict = ee.Dictionary(analysisYearByCountry)                                                                                      
                                                                                                                                    
def assign_year(feature):                                                                                                             
    year = ee.Number(year_dict.get(feature.getString('NAME_0'), ee.Number(2018)))                                                     
    return feature.set('analysis_year', year)                                                                                         
                
GADM_toSample = GADM_toSample.map(assign_year)   

# print countries not in the dictionary
countries_in_GADM = GADM_toSample.aggregate_array('NAME_0').distinct().getInfo()
countries_not_in_dict = set(countries_in_GADM).difference(set(analysisYearByCountry.keys()))
# If empty, print all good, else print the list of countries
if not countries_not_in_dict:
    print("All countries in GADM_toSample have an assigned analysis year.")
else:
    print("The following countries in GADM_toSample do NOT have an assigned analysis year:")
    for country in countries_not_in_dict:
        print(country)

All countries in GADM_toSample have an assigned analysis year.


In [114]:
# For DHS survey we can use the year "SVYYEAR" and for MICS "SVYYEARS" which define the start year of the survey collection.
# Keep source-specific country keys so the same country can appear more than once.

dhs_country_years = dhs_gdf.groupby('CNTRYNAMEE')['SVYYEAR'].first().apply(int).to_dict()
mics_country_years = mics_gdf.groupby('CNTRYNAME')['SVYYEARS'].first().apply(int).to_dict()
wfp_country_years = {'Pakistan': 2022}

analysisYearByCountrySource = {}

for country, year in dhs_country_years.items():
    analysisYearByCountrySource[f"{country}_DHS"] = year

for country, year in mics_country_years.items():
    analysisYearByCountrySource[f"{country}_MICS"] = year

for country, year in wfp_country_years.items():
    analysisYearByCountrySource[f"{country}_WFP"] = year

sortedAnalysisYearByCountrySource = dict(sorted(analysisYearByCountrySource.items()))

for country_source, year in sortedAnalysisYearByCountrySource.items():
    print(f"{country_source}: {year}")

Afghanistan_DHS: 2015
Albania_DHS: 2017
Angola_DHS: 2023
Armenia_DHS: 2016
Bangladesh_DHS: 2022
Benin_DHS: 2017
Bolivia_DHS: 2008
Burkina Faso_DHS: 2021
Burundi_DHS: 2016
Cambodia_DHS: 2021
Cameroon_DHS: 2022
Chad_DHS: 2014
Colombia_DHS: 2015
Comoros_MICS: 2022
Congo Democratic Republic_DHS: 2023
Cote d'Ivoire_DHS: 2021
Eswatini_MICS: 2021
Ethiopia_DHS: 2019
Fiji_MICS: 2021
Gabon_DHS: 2019
Gambia_MICS: 2018
Georgia_MICS: 2018
Ghana_DHS: 2022
Ghana_MICS: 2017
Guatemala_DHS: 2015
Guinea_DHS: 2021
Haiti_DHS: 2016
Honduras_MICS: 2019
Indonesia_DHS: 2017
Jordan_DHS: 2023
Kenya_DHS: 2022
Kosovo_MICS: 2019
Lao People's Democratic Republic_MICS: 2017
Lesotho_DHS: 2023
Lesotho_MICS: 2018
Liberia_DHS: 2019
Madagascar_DHS: 2021
Madagascar_MICS: 2018
Malawi_DHS: 2024
Malawi_MICS: 2019
Maldives_DHS: 2016
Mali_DHS: 2023
Mauritania_DHS: 2020
Mozambique_DHS: 2022
Myanmar_DHS: 2016
Nepal_DHS: 2022
Niger_DHS: 2021
Nigeria_DHS: 2024
Nigeria_MICS: 2016
Pakistan_DHS: 2017
Pakistan_WFP: 2022
Philippines_DHS

In [115]:
bandNames = [
    'Abs_Lat',
    'CGIAR_Aridity_Index',
    'CGIAR_PET',
    'CHELSA_BIO_Annual_Mean_Temperature',
    'CHELSA_BIO_Annual_Precipitation',
    'CHELSA_BIO_Precipitation_Seasonality',
    'CHELSA_BIO_Precipitation_of_Coldest_Quarter',
    'CHELSA_BIO_Precipitation_of_Driest_Month',
    'CHELSA_BIO_Precipitation_of_Driest_Quarter',
    'CHELSA_BIO_Precipitation_of_Warmest_Quarter',
    'CHELSA_BIO_Precipitation_of_Wettest_Month',
    'CHELSA_BIO_Precipitation_of_Wettest_Quarter',
    'CHELSA_BIO_Temperature_Annual_Range',
    'CHELSA_BIO_Temperature_Seasonality',
    'CIFOR_TropicalPeatlandExtent',
    'CSP_Global_Human_Modification',
    'ConsensusLandCoverClass_Barren',
    'ConsensusLandCoverClass_Cultivated_and_Managed_Vegetation',
    'ConsensusLandCoverClass_Deciduous_Broadleaf_Trees',
    'ConsensusLandCoverClass_Evergreen_Broadleaf_Trees',
    'ConsensusLandCoverClass_Evergreen_Deciduous_Needleleaf_Trees',
    'ConsensusLandCoverClass_Herbaceous_Vegetation',
    'ConsensusLandCoverClass_Mixed_Other_Trees',
    'ConsensusLandCoverClass_Open_Water',
    'ConsensusLandCoverClass_Regularly_Flooded_Vegetation',
    'ConsensusLandCoverClass_Shrubs',
    'ConsensusLandCoverClass_Snow_Ice',
    'ConsensusLandCoverClass_Urban_Builtup',
    'ConsensusLandCover_Human_Development_Percentage',
    'CrowtherLab_Tree_Density',
    'EarthEnvCloudCover_CloudForestPrediction',
    'EarthEnvTexture_CoOfVar_EVI',
    'EarthEnvTexture_Contrast_EVI',
    'EarthEnvTexture_Correlation_EVI',
    'EarthEnvTexture_Dissimilarity_EVI',
    'EarthEnvTexture_Entropy_EVI',
    'EarthEnvTexture_Evenness_EVI',
    'EarthEnvTexture_Homogeneity_EVI',
    'EarthEnvTexture_Maximum_EVI',
    'EarthEnvTexture_Range_EVI',
    'EarthEnvTexture_Shannon_Index',
    'EarthEnvTexture_Simpson_Index',
    'EarthEnvTexture_Std_EVI',
    'EarthEnvTexture_Uniformity_EVI',
    'EarthEnvTexture_Variance_EVI',
    'EarthEnvTopoMed_1stOrderPartialDerivEW',
    'EarthEnvTopoMed_1stOrderPartialDerivNS',
    'EarthEnvTopoMed_Eastness',
    'EarthEnvTopoMed_Elevation',
    'EarthEnvTopoMed_Roughness',
    'EarthEnvTopoMed_Slope',
    'EarthEnvTopoMed_TerrainRuggednessIndex',
    'EarthEnvTopoMed_TopoPositionIndex',
    'EsaCci_BurntAreasProbability',
    'FanEtAl_Depth_to_Water_Table_AnnualMean',
    'FanEtAl_Depth_to_Water_Table_AnnualSD',
    'GHS_Population_Density',
    'GPWv4_Population_Density',
    'GiriEtAl_MangrovesExtent',
    'GLW3_RuminantsDistribution_downsampled10km',
    'MODIS_EVI',
    'MODIS_NDVI',
    'MODIS_NPP',
    'PelletierEtAl_SoilAndSedimentaryDepositThicknesses',
    'SG_Absolute_depth_to_bedrock',
    'SG_Bulk_density_015cm',
    'SG_Depth_to_bedrock',
    'SG_H2O_Capacity_015cm',
    'SG_Saturated_H2O_Content_015cm',
    'TootchiEtAl_WetlandsRegularlyFlooded',
    'WCS_Human_Footprint_2009']

In [116]:
# Stack of environmental covariates to sample 
composite = ee.Image.cat([
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_bioComposite_30ArcSec"),
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_climateComposite_30ArcSec"),
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_geoComposite_30ArcSec"),
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_processComposite_30ArcSec"),
        ])

crs = composite.projection().getInfo()['crs']
transform = composite.projection().getInfo()['transform']
print(f"CRS: {crs}")
print(f"Transform: {transform}")

# DYNAMIC COMPOSITE
# Dynamic covariates to sample based on survey year
WorldPop = ee.ImageCollection("projects/sat-io/open-datasets/WORLDPOP/pop")
VIIRS_V21 = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V21")
VIIRS_V22 = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V22")
VIIRS_Annual = VIIRS_V21.merge(VIIRS_V22)
CHIRPS_Daily_Precipitation= ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
MODIS_NDVI_EVI = ee.ImageCollection('MODIS/061/MOD13Q1')
JRC_Built_Up = ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_S")
JRC_DegreeUrban = ee.ImageCollection("JRC/GHSL/P2023A/GHS_SMOD_V2-0")
JRC_Building_Height = ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_H")
JRC_Population = ee.ImageCollection("JRC/GHSL/P2023A/GHS_POP")
MAP_FrictionSurface = ee.Image("projects/malariaatlasproject/assets/accessibility/friction_surface/2019_v5_1")
ERA5_Monthly_Temp = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

# STATIC COMPOSITE
staticComposite = composite.select(bandNames).addBands(MAP_FrictionSurface.select('friction').rename('map_friction'))

# rename JRC building height image
JRC_Building_Height = ee.Image("JRC/GHSL/P2023A/GHS_BUILT_H/2018") \
    .select('built_height') \
    .rename('jrc_building_height')

# Add it as a new band
staticComposite = staticComposite.addBands(JRC_Building_Height)

CRS: EPSG:4326
Transform: [0.008333333333333333, 0, -180, 0, -0.008333333333333333, 88]


In [142]:
# Sample static and dynamic covariates for a single feature
def sampleFeature(feature):
    f = ee.Feature(feature)
    geom = f.geometry()
    reducer = ee.Reducer.mean()

    static_dict = staticComposite.reduceRegion(
        reducer=reducer,
        crs=crs,
        crsTransform=transform,
        geometry=geom,
        maxPixels=1e13,
        tileScale=16
    )

    dynamic_dict = getDynamicImage(ee.Number(f.get('analysis_year'))).reduceRegion(
        reducer=reducer,
        crs=crs,
        crsTransform=transform,
        geometry=geom,
        maxPixels=1e13,
        tileScale=16
    )

    ghsl_pop = getGHSLPopulationInterpolated(ee.Number(f.get('analysis_year')))
    ghsl_pop_proj = ghsl_pop.projection()

    ghsl_pop_sum_dict = ghsl_pop.rename('ghsl_population_sum').reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        crs=ghsl_pop_proj,
        scale=ghsl_pop_proj.nominalScale(),
        maxPixels=1e13,
        tileScale=16
    )

    wp = getWorldPopImage(ee.Number(f.get('analysis_year')))

    worldpop_sum_dict = wp.rename('worldpop_sum').reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        scale=100,
        maxPixels=1e13,
        tileScale=16
    )

    return f.set(static_dict).set(dynamic_dict).set(ghsl_pop_sum_dict).set(worldpop_sum_dict)

# GHSL built surface: linearly interpolate between 5-year intervals
# 2015-2019 -> between 2015 and 2020
# 2020-2024 -> between 2020 and 2025
def getGHSLBuiltInterpolated(year):
    year = ee.Number(year)

    y0 = year.divide(5).floor().multiply(5).max(2015).min(2020)
    y1 = y0.add(5)
    w = year.subtract(y0).divide(5)

    img0 = ee.Image(
        JRC_Built_Up.filterDate(ee.Date.fromYMD(y0, 1, 1), ee.Date.fromYMD(y0.add(1), 1, 1)).first()
    ).select('built_surface')

    img1 = ee.Image(
        JRC_Built_Up.filterDate(ee.Date.fromYMD(y1, 1, 1), ee.Date.fromYMD(y1.add(1), 1, 1)).first()
    ).select('built_surface')

    return img0.add(img1.subtract(img0).multiply(w)).rename('ghsl_built_surface')

# GHSL population: linearly interpolate between 5-year intervals
def getGHSLPopulationInterpolated(year):
    year = ee.Number(year)

    y0 = year.divide(5).floor().multiply(5).max(1975).min(2025)
    y1 = y0.add(5).min(2030)

    w = ee.Number(
        ee.Algorithms.If(y1.eq(y0), 0, year.subtract(y0).divide(y1.subtract(y0)))
    )

    img0 = ee.Image(
        JRC_Population.filterDate(
            ee.Date.fromYMD(y0, 1, 1),
            ee.Date.fromYMD(y0.add(1), 1, 1)
        ).first()
    ).select('population_count')

    img1 = ee.Image(
        JRC_Population.filterDate(
            ee.Date.fromYMD(y1, 1, 1),
            ee.Date.fromYMD(y1.add(1), 1, 1)
        ).first()
    ).select('population_count')

    return img0.add(img1.subtract(img0).multiply(w)).rename('ghsl_population')

# GHSL Degree of Urbanisation:
def getGHSLUrbanFraction(year):
    year = ee.Number(year)

    y0 = year.divide(5).floor().multiply(5).max(2015).min(2020)

    smod = ee.Image(
        JRC_DegreeUrban.filterDate(ee.Date.fromYMD(y0, 1, 1), ee.Date.fromYMD(y0.add(1), 1, 1)).first()
    ).select('smod_code')

    return smod.gte(21).rename('ghsl_urban_frac') # classifies 21, 22, 23, and 30 as urban and everything below 21 as non-urban
    

def getWorldPopImage(year):
    year = ee.Number(year).int()

    wp = (WorldPop
          .filter(ee.Filter.calendarRange(year, year, 'year'))
          .select('population')
          .mosaic()
          .rename('worldpop'))

    return wp
    
    # Annual dynamic composite image 
def getDynamicImage(year):
    year = ee.Number(year).int()

    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')

    # Returns a fully masked image with given bands — used as fallback when a collection is empty
    def masked_fallback(bands):
        if isinstance(bands, str):
            bands = [bands]
        return ee.Image.constant([0] * len(bands)).rename(bands).updateMask(ee.Image.constant(0))

    # WorldPop: annual population image mosaicked across all country tiles
    worldpop = getWorldPopImage(year)

    # VIIRS annual average radiance (starts 2012; pre-2012 surveys return NaN)
    viirs = ee.Image(
        VIIRS_Annual.filterDate(start, end)
        .merge(ee.ImageCollection([masked_fallback(['average'])]))
        .first()
    ).select('average').rename('viirs_average')

    # CHIRPS annual precipitation sum
    chirps_annual_precipitation = CHIRPS_Daily_Precipitation \
        .filterDate(start, end) \
        .merge(ee.ImageCollection([masked_fallback(['precipitation'])])) \
        .select('precipitation') \
        .sum() \
        .rename('chirps_annual_precipitation')

    # MODIS annual mean NDVI and EVI
    modis = MODIS_NDVI_EVI \
        .filterDate(start, end) \
        .merge(ee.ImageCollection([masked_fallback(['NDVI', 'EVI'])])) \
        .select(['NDVI', 'EVI']) \
        .mean() \
        .rename(['modis_ndvi', 'modis_evi'])

    # GHSL built-up surface and population (interpolated)
    ghslBuilt = getGHSLBuiltInterpolated(year)

    ghslPopulation = getGHSLPopulationInterpolated(year)

    # GHSL urban fraction
    ghslUrban = getGHSLUrbanFraction(year)

    # ERA5: merge fallback once, reuse for all bands
    era5_with_fallback = ERA5_Monthly_Temp \
        .filterDate(start, end) \
        .merge(ee.ImageCollection([masked_fallback(['temperature_2m', 'runoff_max', 'runoff_min', 'temperature_2m_max'])]))

    era5_temperature_2m = era5_with_fallback \
        .select('temperature_2m').mean().rename('era5_temperature_2m')

    runoff_max_annualmax = era5_with_fallback \
        .select('runoff_max').max().rename('runoff_max_annualmax')

    runoff_min_annualmin = era5_with_fallback \
        .select('runoff_min').min().rename('runoff_min_annualmin')

    temperature_2m_max_annualmax = era5_with_fallback \
        .select('temperature_2m_max').max().rename('temperature_2m_max_annualmax')

    return ee.Image.cat([
        worldpop,
        viirs,
        chirps_annual_precipitation,
        modis,
        ghslBuilt,
        ghslPopulation,
        ghslUrban,
        era5_temperature_2m,
        runoff_max_annualmax,
        runoff_min_annualmin,
        temperature_2m_max_annualmax
    ])



In [119]:
 # Convert DHS and MICS geodataframes to feature collections
 dhs_fc = geemap.geopandas_to_ee(dhs_gdf, geodesic=False)
 mics_fc = geemap.geopandas_to_ee(mics_gdf, geodesic=False)
 wfp_fc = geemap.geopandas_to_ee(wfp_gdf, geodesic=False)
 gadm_lmic_2024_fc = geemap.geopandas_to_ee(gadm_lmic_gdf, geodesic=False)


 #write to shapefile, if file doesn't exist yet

os.makedirs("data", exist_ok=True)

def prepare_for_export(gdf):
    out = gdf.copy()

    # Earth Engine recommends EPSG:4326 for uploaded vectors
    if out.crs is not None and out.crs.to_string() != "EPSG:4326":
        out = out.to_crs("EPSG:4326")

    # make sure IDs are strings if needed
    if "GEO_ID" in out.columns:
        out["GEO_ID"] = out["GEO_ID"].astype(str)

        # Shorten long field names for shapefile compatibility
    if "analysis_year" in out.columns:
        out = out.rename(columns={"analysis_year": "anl_year"})


    return out

files_to_write = [
    (dhs_gdf, "outputs/02_boundary_files/dhs_fc_cleaned_tmp.shp"),
    (mics_gdf, "outputs/02_boundary_files/mics_fc_cleaned_tmp.shp"),
    (wfp_gdf, "outputs/02_boundary_files/wfp_fc_cleaned_tmp.shp"),
    (gadm_lmic_gdf, "outputs/02_boundary_files/gadm_lmic_fc_cleaned_tmp_v2.shp"),
]

for gdf, path in files_to_write:
    if not os.path.exists(path):
        prepare_for_export(gdf).to_file(path)

# Manual step: upload to GEE 


<positron-console-cell-119>:39: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'admin_level' to 'admin_leve'
  ogr_write(


In [126]:
dhs_fc = ee.FeatureCollection("projects/ee-estheregreenwood/assets/2026_worldbank/dhs_fc_cleaned_tmp")
mics_fc = ee.FeatureCollection("projects/ee-estheregreenwood/assets/2026_worldbank/mics_fc_cleaned_tmp")
wfp_fc = ee.FeatureCollection("projects/ee-estheregreenwood/assets/2026_worldbank/wfp_fc_cleaned_tmp")
gadm_fc = ee.FeatureCollection("projects/ee-estheregreenwood/assets/2026_worldbank/gadm_lmic_fc_cleaned_tmp_v2")

# rename anl_year back to analysis_year
dhs_fc = dhs_fc.map(lambda f: f.set('analysis_year', f.get('analysis_y'))) ## check how analysis_y is coded in your file
mics_fc = mics_fc.map(lambda f: f.set('analysis_year', f.get('analysis_y')))
wfp_fc = wfp_fc.map(lambda f: f.set('analysis_year', f.get('analysis_y')))
gadm_fc = gadm_fc.map(lambda f: f.set('analysis_year', f.get('anl_year')))

dhs_fc = dhs_fc.map(lambda f: f.set('admin_level', f.get('admin_leve')))
mics_fc = mics_fc.map(lambda f: f.set('admin_level', f.get('admin_leve')))
wfp_fc = wfp_fc.map(lambda f: f.set('admin_level', f.get('admin_leve')))
gadm_fc = gadm_fc.map(lambda f: f.set('admin_level', f.get('admin_leve')))

# export both to GEE
export_dhs = ee.batch.Export.table.toAsset(
    collection=dhs_fc,
    description='dhs_fc_cleaned_v2',
    assetId='projects/ee-estheregreenwood/assets/2026_worldbank/dhs_fc_cleaned_v2'
)
export_dhs.start()  

export_mics = ee.batch.Export.table.toAsset(
    collection=mics_fc,
    description='mics_fc_cleaned_v2',
    assetId='projects/ee-estheregreenwood/assets/2026_worldbank/mics_fc_cleaned_v2'
)
export_mics.start()

export_wfp = ee.batch.Export.table.toAsset(
    collection=wfp_fc,
    description='wfp_fc_cleaned_v2',
    assetId='projects/ee-estheregreenwood/assets/2026_worldbank/wfp_fc_cleaned_v2'
)
export_wfp.start()

export_gadm = ee.batch.Export.table.toAsset(
    collection=gadm_fc,
    description='gadm_fc_cleaned_v2',
    assetId='projects/ee-estheregreenwood/assets/2026_worldbank/gadm_fc_cleaned_v2'
)
export_gadm.start()

In [127]:
dhs_fc = ee.FeatureCollection('projects/ee-estheregreenwood/assets/2026_worldbank/dhs_fc_cleaned_v2')
mics_fc = ee.FeatureCollection('projects/ee-estheregreenwood/assets/2026_worldbank/mics_fc_cleaned_v2')
wfp_fc = ee.FeatureCollection('projects/ee-estheregreenwood/assets/2026_worldbank/wfp_fc_cleaned_v2')
gadm_fc = ee.FeatureCollection('projects/ee-estheregreenwood/assets/2026_worldbank/gadm_fc_cleaned_v2')


# Final dataset to sample: GADM_toSample, dhs_fc, and mics_fc, wfp merged
train_fc = ee.FeatureCollection([GADM_toSample, dhs_fc, mics_fc, wfp_fc]).flatten()
# Prediction data only
pred_fc = gadm_fc

print(f"Total number of training features to sample: {train_fc.size().getInfo()}")
print(f"Total number of prediction features to sample: {pred_fc.size().getInfo()}")


Total number of training features to sample: 1476
Total number of prediction features to sample: 2310


In [131]:
status = export_task.status()
print(status)

{'state': 'RUNNING', 'description': 'worldbank_sampled_pred_v2', 'priority': 100, 'creation_timestamp_ms': 1776800361172, 'update_timestamp_ms': 1776842328983, 'start_timestamp_ms': 1776829628751, 'task_type': 'EXPORT_FEATURES', 'attempt': 2, 'batch_eecu_usage_seconds': 0.007134146, 'id': 'R33WDUQP2QPXRCXJBXIYWMOZ', 'name': 'projects/ee-estheregreenwood/operations/R33WDUQP2QPXRCXJBXIYWMOZ'}


In [128]:
# Check for non-numeric values that wouldn't show as NaN                                                                              
from numpy import sort

print(dhs_gdf['analysis_year'].dtype)                                                                                                 
print(dhs_gdf['analysis_year'].unique()[:20])        

print(mics_gdf['analysis_year'].dtype)                                                                                                 
print(mics_gdf['analysis_year'].unique()[:20])

print(wfp_gdf['analysis_year'].dtype)                                                                                                 
print(wfp_gdf['analysis_year'].unique()[:20])

print(gadm_lmic_gdf['analysis_year'].dtype)                                                                                                 
print(gadm_lmic_gdf['analysis_year'].unique()[:20])
                                                                                                                                    
# Check GADM features server-side (sample 10)                                                                                         
print(GADM_toSample.aggregate_histogram('analysis_year').keys().getInfo())
print(sorted(dhs_fc.aggregate_histogram('analysis_year').keys().getInfo()))
print(sorted(mics_fc.aggregate_histogram('analysis_year').keys().getInfo()))
print(sorted(wfp_fc.aggregate_histogram('analysis_year').keys().getInfo()))
print(sorted(gadm_fc.aggregate_histogram('analysis_year').keys().getInfo()))

int64
[2023 2016 2019 2015 2020 2017 2022 2021 2024 2014 2008 2018]
int64
[2018 2022 2017 2019 2016 2021]
int64
[2022]
int64
[2024]
['2016', '2018', '2019', '2023']
['2008', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
['2016', '2017', '2018', '2019', '2021', '2022']
['2022']
['2024']


In [129]:
# Check country/year/level combinations across all sources
# Admin level is inferred from the highest NAME_N property that is set and not 'NaN'
def infer_level(feature):
    def is_valid(prop):
        val = ee.String(ee.Algorithms.If(
            feature.propertyNames().contains(prop),
            feature.get(prop), 'NaN'
        ))
        return val.compareTo('NaN').neq(0).And(val.length().gt(0))
    return ee.Number(ee.Algorithms.If(is_valid('NAME_3'), 3,
                     ee.Algorithms.If(is_valid('NAME_2'), 2,
                     ee.Algorithms.If(is_valid('NAME_1'), 1, 0))))

def gadm_row(feature):
    return ee.Feature(None, {
        'country': feature.getString('NAME_0'),
        'year':    ee.Number(feature.get('analysis_year')).int(),
        'source':   infer_level(feature),
        'region':  feature.getString('HH7')
    })

def gadm_sample_row(feature):
    return ee.Feature(None, {
        'source':      ee.String('GADM'),
        'country':     feature.getString('NAME_0'),
        'year':        ee.Number(feature.get('analysis_year')).int(),
        'admin_level': infer_level(feature),
        'region':      infer_region_name(feature)
    })


def dhs_row(feature):
    return ee.Feature(None, {
        'country': feature.getString('CNTRYNAMEE'),
        'year':    ee.Number(feature.get('analysis_year')).int(),
        'source':   ee.String('DHS'),
        'admin_level': None,
        'region':  feature.getString('DHSREGEN')
    })

def mics_row(feature):
    return ee.Feature(None, {
        'country': feature.getString('CNTRYNAME'),
        'year':    ee.Number(feature.get('analysis_year')).int(),
        'source':   ee.String('MICS'),
        'admin_level': None,
        'region':  feature.getString('REGNAME')
    })

def wfp_row(feature):
    return ee.Feature(None, {
        'country': feature.getString('adm0_name'),
        'year':    ee.Number(feature.get('analysis_year')).int(),
        'source':   ee.String('WFP'),
        'admin_level': None,
        'region':  feature.getString('adm2_name')
    })

country_years = ee.FeatureCollection([
    GADM_toSample.map(gadm_row).distinct(['country', 'year', 'source']),
    dhs_fc.map(dhs_row).distinct(['country', 'year', 'source']),
    mics_fc.map(mics_row).distinct(['country', 'year', 'source']),
    wfp_fc.map(wfp_row).distinct(['country', 'year', 'source'])
]).flatten()

df = geemap.ee_to_df(country_years) \
          .sort_values(['source', 'country']) \
          .reset_index(drop=True)

# Flag countries whose year doesn't match analysisYearByCountry
df['expected_year'] = df['country'].map(analysisYearByCountry)
df['ok'] = df.apply(lambda r: r['year'] == r['expected_year'] or pd.isna(r['expected_year']), axis=1)
print(df[['source','country','year','expected_year','ok']].to_string())


   source                           country  year  expected_year     ok
0       1                           Algeria  2018         2018.0   True
1       1          Central African Republic  2018         2018.0   True
2       1                     Guinea-Bissau  2018         2018.0   True
3       1                            Guyana  2019         2019.0   True
4       1                              Iraq  2018         2018.0   True
5       1                          Kiribati  2018         2018.0   True
6       1                              Laos  2023         2023.0   True
7       1                          Mongolia  2018         2018.0   True
8       1                          Paraguay  2016         2016.0   True
9       1                          Suriname  2018         2018.0   True
10      1             São Tomé and Príncipe  2019         2019.0   True
11      1                             Tonga  2019         2019.0   True
12      1                           Tunisia  2023         2023.0

In [151]:
# ------------------------------------------------------------
# pred_fc is intentional
# Set analysis_year = 2024 for all features
# ------------------------------------------------------------
pred_fc_2024 = pred_fc.map(lambda f: ee.Feature(f).set('analysis_year', 2024))


# ------------------------------------------------------------
# Helper: WorldPop image for a given year
# ------------------------------------------------------------
def getWorldPopImage(year):
    year = ee.Number(year).int()

    wp = (WorldPop
          .filter(ee.Filter.calendarRange(year, year, 'year'))
          .select('population')
          .mosaic()
          .rename('worldpop'))

    return wp


# ------------------------------------------------------------
# Helper: GHSL population interpolated for a given year
# ------------------------------------------------------------
def getGHSLPopulationInterpolated(year):
    year = ee.Number(year)

    y0 = year.divide(5).floor().multiply(5).max(1975).min(2025)
    y1 = y0.add(5).min(2030)

    w = ee.Number(
        ee.Algorithms.If(y1.eq(y0), 0, year.subtract(y0).divide(y1.subtract(y0)))
    )

    img0 = ee.Image(
        JRC_Population.filterDate(
            ee.Date.fromYMD(y0, 1, 1),
            ee.Date.fromYMD(y0.add(1), 1, 1)
        ).first()
    ).select('population_count')

    img1 = ee.Image(
        JRC_Population.filterDate(
            ee.Date.fromYMD(y1, 1, 1),
            ee.Date.fromYMD(y1.add(1), 1, 1)
        ).first()
    ).select('population_count')

    return img0.add(img1.subtract(img0).multiply(w)).rename('ghsl_population')


# ------------------------------------------------------------
# Precompute the 2024 rasters once
# ------------------------------------------------------------
wp_2024 = getWorldPopImage(2024)
ghsl_pop_2024 = getGHSLPopulationInterpolated(2024)
ghsl_pop_2024_proj = ghsl_pop_2024.projection()


# ------------------------------------------------------------
# Sample only population sums
# ------------------------------------------------------------
def samplePopulationSums2024(feature):
    f = ee.Feature(feature)
    geom = f.geometry()

    ghsl_pop_sum_dict = ghsl_pop_2024.rename('ghsl_population_sum').reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        crs=ghsl_pop_2024_proj,
        scale=ghsl_pop_2024_proj.nominalScale(),
        maxPixels=1e13,
        tileScale=16
    )

    worldpop_sum_dict = wp_2024.rename('worldpop_sum').reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        scale=100,
        maxPixels=1e13,
        tileScale=16
    )

    return f.set('analysis_year', 2024).set(ghsl_pop_sum_dict).set(worldpop_sum_dict)



In [147]:
# ------------------------------------------------------------
pred_fc_population_sums_2024 = pred_fc_2024.map(samplePopulationSums2024)

# ------------------------------------------------------------
# Export directly to CSV on Google Drive
# ------------------------------------------------------------
export_task = ee.batch.Export.table.toDrive(
    collection=pred_fc_population_sums_2024,
    description='pred_fc_population_sums_2024_csv',
    folder='earthengine_exports',
    fileNamePrefix='pred_fc_population_sums_2024',
    fileFormat='CSV',
    selectors=[
        'NAME_0',
        'NAME_1',
        'analysis_year',
        'worldpop_sum',
        'ghsl_population_sum'
    ]
)

export_task.start()
print(export_task.status())

{'state': 'READY', 'description': 'pred_fc_population_sums_2024_csv', 'priority': 100, 'creation_timestamp_ms': 1776862372398, 'update_timestamp_ms': 1776862372398, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'V2VW3TAZIDUG52WCM7FISL22', 'name': 'projects/ee-estheregreenwood/operations/V2VW3TAZIDUG52WCM7FISL22'}


In [153]:
import ee
import re
import time
import unicodedata

# ------------------------------------------------------------
# pred_fc is intentional
# Set analysis_year = 2024 for all features
# ------------------------------------------------------------
pred_fc_2024 = pred_fc.map(lambda f: ee.Feature(f).set('analysis_year', 2024))


# ------------------------------------------------------------
# Helper: WorldPop image for a given year
# ------------------------------------------------------------
def getWorldPopImage(year):
    year = ee.Number(year).int()

    wp = (WorldPop
          .filter(ee.Filter.calendarRange(year, year, 'year'))
          .select('population')
          .mosaic()
          .rename('worldpop'))

    return wp


# ------------------------------------------------------------
# Helper: GHSL population interpolated for a given year
# ------------------------------------------------------------
def getGHSLPopulationInterpolated(year):
    year = ee.Number(year)

    y0 = year.divide(5).floor().multiply(5).max(1975).min(2025)
    y1 = y0.add(5).min(2030)

    w = ee.Number(
        ee.Algorithms.If(y1.eq(y0), 0, year.subtract(y0).divide(y1.subtract(y0)))
    )

    img0 = ee.Image(
        JRC_Population.filterDate(
            ee.Date.fromYMD(y0, 1, 1),
            ee.Date.fromYMD(y0.add(1), 1, 1)
        ).first()
    ).select('population_count')

    img1 = ee.Image(
        JRC_Population.filterDate(
            ee.Date.fromYMD(y1, 1, 1),
            ee.Date.fromYMD(y1.add(1), 1, 1)
        ).first()
    ).select('population_count')

    return img0.add(img1.subtract(img0).multiply(w)).rename('ghsl_population')


# ------------------------------------------------------------
# Precompute the 2024 rasters once
# ------------------------------------------------------------
wp_2024 = getWorldPopImage(2024)
ghsl_pop_2024 = getGHSLPopulationInterpolated(2024)
ghsl_pop_2024_proj = ghsl_pop_2024.projection()


# ------------------------------------------------------------
# Sample only population sums
# ------------------------------------------------------------
def samplePopulationSums2024(feature):
    f = ee.Feature(feature)
    geom = f.geometry()

    ghsl_pop_sum_dict = ghsl_pop_2024.rename('ghsl_population_sum').reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        crs=ghsl_pop_2024_proj,
        scale=ghsl_pop_2024_proj.nominalScale(),
        maxPixels=1e13,
        tileScale=16
    )

    worldpop_sum_dict = wp_2024.rename('worldpop_sum').reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        scale=100,
        maxPixels=1e13,
        tileScale=16
    )

    return (f
            .set('analysis_year', 2024)
            .set(ghsl_pop_sum_dict)
            .set(worldpop_sum_dict))


# ------------------------------------------------------------
# Helper: safe file/task names
# ------------------------------------------------------------
def slugify(text):
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[^A-Za-z0-9]+', '_', text).strip('_')
    return text or 'country'


# ------------------------------------------------------------
# Get unique country names from pred_fc
# ------------------------------------------------------------
countries = sorted(pred_fc_2024.aggregate_array('NAME_0').distinct().getInfo())
print(f"Found {len(countries)} countries")
print(countries[:10])


# ------------------------------------------------------------
# Choose which countries to submit
# Set COUNTRY_END = None to submit all remaining countries
# ------------------------------------------------------------
COUNTRY_START = 0
COUNTRY_END = 20

countries_to_run = countries[COUNTRY_START:COUNTRY_END]
print(f"Submitting {len(countries_to_run)} country exports")


# ------------------------------------------------------------
# Optional: quick 5-feature validation before exports
# ------------------------------------------------------------
validation_fc_5 = (
    pred_fc_2024
    .randomColumn('rand', seed=42)
    .sort('rand')
    .limit(5)
)

feat_list = validation_fc_5.toList(5)

for i in range(5):
    feat = ee.Feature(feat_list.get(i))
    out = samplePopulationSums2024(feat).toDictionary([
        'NAME_0',
        'NAME_1',
        'analysis_year',
        'worldpop_sum',
        'ghsl_population_sum'
    ]).getInfo()

    print(f"\nValidation feature {i+1}")
    print(out)


# ------------------------------------------------------------
# Submit one export task per country
# ------------------------------------------------------------
tasks = []

for country in countries_to_run:
    print(f"Preparing export for: {country}")

    country_fc = pred_fc_2024.filter(ee.Filter.eq('NAME_0', country))
    sampled_country_fc = country_fc.map(samplePopulationSums2024)

    safe_country = slugify(country)

    task = ee.batch.Export.table.toDrive(
        collection=sampled_country_fc,
        description=f'pred_fc_pop_sums_2024_{safe_country}',
        folder='earthengine_exports_pop_sums_2024',
        fileNamePrefix=f'pred_fc_pop_sums_2024_{safe_country}',
        fileFormat='CSV',
        selectors=[
            'NAME_0',
            'NAME_1',
            'analysis_year',
            'worldpop_sum',
            'ghsl_population_sum'
        ]
    )

    task.start()
    tasks.append((country, task))
    print(f"Started task for {country}: {task.id}")

    # Small pause so task creation is gentler
    time.sleep(0.2)


# ------------------------------------------------------------
# Print initial task states
# ------------------------------------------------------------
print("\nInitial task statuses:")
for country, task in tasks:
    status = task.status()
    print(country, status.get('state'), status.get('description'))

Found 130 countries
['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Argentina', 'Armenia', 'Azerbaijan', 'Bangladesh', 'Belarus', 'Belize']
Submitting 20 country exports

Validation feature 1
{'NAME_0': 'Colombia', 'NAME_1': 'Nariño', 'analysis_year': 2024, 'ghsl_population_sum': 1687481.8481642795, 'worldpop_sum': 1361666.7141652093}

Validation feature 2
{'NAME_0': 'Dominica', 'NAME_1': 'Saint John', 'analysis_year': 2024, 'ghsl_population_sum': 7461.476944575683, 'worldpop_sum': 5289.982507526944}

Validation feature 3
{'NAME_0': 'Argentina', 'NAME_1': 'Salta', 'analysis_year': 2024, 'ghsl_population_sum': 1499298.5479328898, 'worldpop_sum': 1257108.626295803}

Validation feature 4
{'NAME_0': 'Lesotho', 'NAME_1': 'Leribe', 'analysis_year': 2024, 'ghsl_population_sum': 353091.6953747867, 'worldpop_sum': 337296.30614414334}

Validation feature 5
{'NAME_0': 'Afghanistan', 'NAME_1': 'Takhar', 'analysis_year': 2024, 'ghsl_population_sum': 1677833.505415258, 'worldpop_sum': 1258392.40417

In [130]:
# Sample variables

#worldbank_sampled_train = train_fc.map(sampleFeature)

#export_task = ee.batch.Export.table.toAsset(
    #collection = worldbank_sampled_train,
    #description = 'worldbank_sampled_train',
    #assetId = 'projects/ee-estheregreenwood/assets/2026_worldbank/worldbank_sampled_train'
#)
#export_task.start()

worldbank_sampled_pred = pred_fc.map(sampleFeature)

export_task = ee.batch.Export.table.toAsset(
    collection = worldbank_sampled_pred,
    description = 'worldbank_sampled_pred_v2',
    assetId = 'projects/ee-estheregreenwood/assets/2026_worldbank/worldbank_sampled_pred_v2'
)
export_task.start()

In [ ]:
# After export is finished, load the sampled data as a feature collection and convert to geodataframe
worldbank_sampled_pred_fc = ee.FeatureCollection('projects/ee-estheregreenwood/assets/2026_worldbank/worldbank_sampled_pred')
worldbank_sampled_pred_gdf = geemap.ee_to_gdf(worldbank_sampled_pred_fc)
worldbank_sampled_pred_gdf.head()
worldbank_sampled_pred_gdf.to_csv('outputs/03_sampled_features/worldbank_sampled_pred_v2.csv', index=False)

In [ ]:
worldbank_sampled_gdf.head()


,geometry,Abs_Lat,CC_1,CC_2,CC_3,CGIAR_Aridity_Index,CGIAR_PET,CHELSA_BIO_Annual_Mean_Temperature,CHELSA_BIO_Annual_Precipitation,CHELSA_BIO_Precipitation_Seasonality,...,era5_temperature_2m,ghsl_built_surface,ghsl_urban_frac,map_friction,modis_evi,modis_ndvi,runoff_max_annualmax,runoff_min_annualmin,temperature_2m_max_annualmax,viirs_average
0,"MULTIPOLYGON (((43.62283 -12.28618, 43.62306 -...",12.323293,None,None,None,10309.259542,2007.328500,250.282019,1342.349195,67.906132,...,298.924320,1.982414,0.189731,0.037722,4664.112468,6795.885880,0.007733,-1.490116e-08,302.925232,0.268968
1,"MULTIPOLYGON (((44.19802 -12.16507, 44.19807 -...",12.222260,None,None,None,9394.301269,1978.478524,237.320639,1955.272813,67.012768,...,297.613088,40.870576,0.039374,0.043836,4191.513321,6537.058973,0.003740,1.142953e-06,304.112908,0.296196
2,"POLYGON ((43.22855 -11.73635, 43.22877 -11.736...",11.662888,None,None,None,12717.777366,1829.624897,226.023979,1787.283132,55.089772,...,296.523902,33.855743,0.176459,0.036054,4021.184823,6614.508839,0.002627,8.889113e-07,302.813624,0.351568
3,"MULTIPOLYGON (((43.62283 -12.28618, 43.62306 -...",12.323293,None,None,None,10309.259542,2007.328500,250.282019,1342.349195,67.906132,...,298.924320,1.982414,0.189731,0.037722,4664.112468,6795.885880,0.007733,-1.490116e-08,302.925232,0.268968
4,"MULTIPOLYGON (((44.19802 -12.16507, 44.19807 -...",12.222260,None,None,None,9394.301269,1978.478524,237.320639,1955.272813,67.012768,...,297.613088,40.870576,0.039374,0.043836,4191.513321,6537.058973,0.003740,1.142953e-06,304.112908,0.296196


In [ ]:
# Coverage check: tally country / source / level / year from worldbank_sampled_gdf

def _nonempty(r, col):
    v = r.get(col, '')
    return pd.notna(v) and str(v).strip() not in ('', 'nan', 'NaN')

def classify_row(r):
    if _nonempty(r, 'CNTRYNAMEE'):
        return str(r['CNTRYNAMEE']), 'DHS', '1'
    if _nonempty(r, 'CNTRYNAME'):
        if _nonempty(r, 'GEONAMET'):   lvl = '3'
        elif _nonempty(r, 'GEONAMES'): lvl = '2'
        elif _nonempty(r, 'GEONAME'):  lvl = '1'
        else:                           lvl = '0'
        return str(r['CNTRYNAME']), 'MICS', lvl
    country = str(r.get('NAME_0', ''))
    if _nonempty(r, 'NAME_3'):   lvl = '3'
    elif _nonempty(r, 'NAME_2'): lvl = '2'
    elif _nonempty(r, 'NAME_1'): lvl = '1'
    else:                         lvl = '0'
    return country, 'GADM', lvl

worldbank_sampled_gdf[['country','source','level']] = pd.DataFrame(
    worldbank_sampled_gdf.apply(classify_row, axis=1).tolist(), index=worldbank_sampled_gdf.index
)

coverage_df = (
    worldbank_sampled_gdf.groupby(['country','source','level'])['analysis_year']
    .apply(lambda s: ', '.join(str(int(y)) for y in sorted(s.dropna().unique())))
    .reset_index()
    .rename(columns={'analysis_year': 'years'})
    .sort_values(['source','country','level'])
    .reset_index(drop=True)
)
print(coverage_df.to_string())
coverage_df.to_csv('worldbank_sampled_coverage.csv', index=False)

                              country source level       years
0                             Albania    DHS     1        2017
1                              Angola    DHS     1        2023
2                             Armenia    DHS     1        2016
3                          Bangladesh    DHS     1        2022
4                               Benin    DHS     1        2017
5                             Bolivia    DHS     1        2008
6                        Burkina Faso    DHS     1        2021
7                             Burundi    DHS     1        2016
8                            Cambodia    DHS     1        2021
9                            Cameroon    DHS     1        2022
10                               Chad    DHS     1        2014
11                           Colombia    DHS     1        2015
12          Congo Democratic Republic    DHS     1        2023
13                      Cote d'Ivoire    DHS     1        2021
14                           Ethiopia    DHS     1  201